# Joint confidence models

Estimate change in the single confidence rating by study condition, overall and within joint judgment strata.

## Load packages

In [10]:
library(readxl)
library(dplyr)
library(lme4)
library(emmeans)

## Build one row per participant × vignette

Transitions use the same four labels as the confidence models. Rows need both transitions and a confidence change. Study condition is coded as in those models (Control = reference). Age and sex are not in the model; rows missing them are kept.

In [11]:
ROOT <- normalizePath("..", winslash = "/")
PATH <- file.path(ROOT, "data", "Psy_Case_Vignette_Data_Prolific_analysis.xlsx")
VIGNETTES <- c("A", "B", "C", "D", "E")
ARM_CODES <- c("4", "1", "2", "3")
ARM_LABELS <- c("Control", "ChatGPT", "Copilot", "Perplexity")
TRANSITION_LEVELS <- c("wrong_right", "right_right", "wrong_wrong", "right_wrong")
COLLAPSED_LEVELS <- c("both_stayed_wrong", "one_stayed_wrong", "neither_stayed_wrong")

classify_transition <- function(pre, post) {
  case_when(
    is.na(pre) | is.na(post) ~ NA_character_,
    pre == 0 & post == 1 ~ "wrong_right",
    pre == 1 & post == 1 ~ "right_right",
    pre == 0 & post == 0 ~ "wrong_wrong",
    pre == 1 & post == 0 ~ "right_wrong",
    TRUE ~ NA_character_
  )
}

df <- read_excel(PATH)

joint <- bind_rows(lapply(VIGNETTES, function(v) {
  df %>%
    transmute(
      prolific_id = as.factor(prolific_id),
      vignette = factor(v, levels = VIGNETTES),
      study_arm = as.character(study_arm),
      condition_transition = classify_transition(
        .data[[paste0("vig_", v, "_pre_condition_correct")]],
        .data[[paste0("vig_", v, "_post_condition_correct")]]
      ),
      triage_transition = classify_transition(
        .data[[paste0("vig_", v, "_pre_nextstep_correct")]],
        .data[[paste0("vig_", v, "_post_nextstep_correct")]]
      ),
      delta_confidence = .data[[paste0("vig_", v, "_post_confidence")]] -
        .data[[paste0("vig_", v, "_pre_confidence")]]
    )
})) %>%
  filter(
    !is.na(delta_confidence),
    !is.na(condition_transition),
    !is.na(triage_transition),
    !is.na(study_arm),
    !is.na(prolific_id)
  ) %>%
  mutate(
    study_arm = factor(study_arm, levels = ARM_CODES, labels = ARM_LABELS),
    study_arm = relevel(study_arm, ref = "Control"),
    condition_transition = factor(condition_transition, levels = TRANSITION_LEVELS),
    triage_transition = factor(triage_transition, levels = TRANSITION_LEVELS),
    collapsed = factor(
      case_when(
        condition_transition == "wrong_wrong" & triage_transition == "wrong_wrong" ~ "both_stayed_wrong",
        condition_transition == "wrong_wrong" | triage_transition == "wrong_wrong" ~ "one_stayed_wrong",
        TRUE ~ "neither_stayed_wrong"
      ),
      levels = COLLAPSED_LEVELS
    )
  )

cat(
  "Rows:", nrow(joint),
  "; participants:", n_distinct(joint$prolific_id),
  "; vignettes:", n_distinct(joint$vignette), "\n"
)
print(table(joint$study_arm))

Rows: 4025 ; participants: 805 ; vignettes: 5 

   Control    ChatGPT    Copilot Perplexity 
      1035       1040       1010        940 


## 4 × 4 counts and mean confidence change

Rows are the condition-identification transition. Columns are the triage transition. Each cell is mean Δconfidence (post − pre), with n in parentheses.

In [12]:
cell_summary <- joint %>%
  group_by(study_arm, condition_transition, triage_transition) %>%
  summarise(
    n = n(),
    mean_delta = mean(delta_confidence),
    .groups = "drop"
  ) %>%
  mutate(cell = sprintf("%.1f (n=%d)", mean_delta, n))

for (arm in levels(joint$study_arm)) {
  cat("\n", arm, "\n", sep = "")
  sub <- cell_summary %>% filter(study_arm == arm)
  wide <- xtabs(n ~ condition_transition + triage_transition, data = sub)
  means <- xtabs(mean_delta ~ condition_transition + triage_transition, data = sub)
  shown <- matrix("", nrow = nrow(wide), ncol = ncol(wide),
                  dimnames = dimnames(wide))
  shown[] <- sprintf("%.1f (n=%d)", means, wide)
  shown[wide == 0] <- "—"
  print(shown, quote = FALSE)
}


Control
                    triage_transition
condition_transition wrong_right right_right wrong_wrong right_wrong
         wrong_right 11.1 (n=31) 8.8 (n=71)  10.6 (n=39) 12.5 (n=8) 
         right_right 13.1 (n=21) 6.8 (n=249) 10.3 (n=71) 5.0 (n=20) 
         wrong_wrong 8.3 (n=38)  3.0 (n=141) 3.9 (n=147) 2.2 (n=52) 
         right_wrong 8.6 (n=8)   4.9 (n=93)  4.2 (n=22)  -4.5 (n=24)

ChatGPT
                    triage_transition
condition_transition wrong_right right_right  wrong_wrong right_wrong
         wrong_right 17.6 (n=78) 20.0 (n=101) 22.5 (n=72) 12.2 (n=16)
         right_right 23.9 (n=42) 12.8 (n=283) 21.9 (n=72) 16.6 (n=32)
         wrong_wrong 10.9 (n=51) 9.9 (n=99)   12.9 (n=63) 6.5 (n=24) 
         right_wrong 5.8 (n=9)   7.6 (n=76)   5.7 (n=14)  -3.2 (n=8) 

Copilot
                    triage_transition
condition_transition wrong_right right_right  wrong_wrong  right_wrong
         wrong_right 15.9 (n=24) 23.9 (n=87)  19.8 (n=38)  8.0 (n=8)  
         right_right 1

## Collapsed cells

- **both_stayed_wrong:** condition and triage were both wrong before and after.
- **one_stayed_wrong:** exactly one of the two judgments stayed wrong.
- **neither_stayed_wrong:** neither judgment stayed wrong.

In [13]:
collapsed_summary <- joint %>%
  group_by(study_arm, collapsed) %>%
  summarise(
    n = n(),
    participants = n_distinct(prolific_id),
    mean_delta = mean(delta_confidence),
    sd_delta = sd(delta_confidence),
    .groups = "drop"
  )

cat("Mean Δconfidence by study condition and collapsed cell\n\n")
print(as.data.frame(collapsed_summary), row.names = FALSE)

cat("\nAmong condition wrong→wrong rows, share that are also triage wrong→wrong:\n")
joint %>%
  filter(condition_transition == "wrong_wrong") %>%
  group_by(study_arm) %>%
  summarise(
    n_condition_wrong_wrong = n(),
    n_also_triage_wrong_wrong = sum(triage_transition == "wrong_wrong"),
    share = mean(triage_transition == "wrong_wrong"),
    .groups = "drop"
  ) %>%
  print(n = Inf)

Mean Δconfidence by study condition and collapsed cell

  study_arm            collapsed   n participants mean_delta sd_delta
    Control    both_stayed_wrong 147           99   3.918367 24.63362
    Control     one_stayed_wrong 363          178   5.746556 23.11553
    Control neither_stayed_wrong 525          196   6.777143 21.89535
    ChatGPT    both_stayed_wrong  63           50  12.873016 20.50766
    ChatGPT     one_stayed_wrong 332          173  14.984940 23.81417
    ChatGPT neither_stayed_wrong 645          205  14.508527 22.42597
    Copilot    both_stayed_wrong 140          100  13.242857 19.88270
    Copilot     one_stayed_wrong 326          172  13.895706 21.85156
    Copilot neither_stayed_wrong 544          195  10.972426 20.62078
 Perplexity    both_stayed_wrong  80           60  11.987500 18.39338
 Perplexity     one_stayed_wrong 303          157  14.858086 23.08341
 Perplexity neither_stayed_wrong 557          187  14.926391 21.88870

Among condition wrong→wrong rows,

## Both-wrong mixed model

Restricted to vignettes where condition identification and triage both stayed wrong. Unadjusted linear mixed model: `delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette)`, REML, Control as reference. Coefficients for ChatGPT, Copilot, and Perplexity are the contrasts versus control.

In [14]:
both_wrong <- joint %>%
  filter(collapsed == "both_stayed_wrong")

cat(
  "Both-wrong rows:", nrow(both_wrong),
  "; participants:", n_distinct(both_wrong$prolific_id), "\n"
)
print(table(both_wrong$study_arm))
cat("\nRows by vignette:\n")
print(table(both_wrong$vignette, both_wrong$study_arm))

both_wrong_model <- lmer(
  delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette),
  data = both_wrong,
  REML = TRUE
)

print(summary(both_wrong_model))

coef_mat <- coef(summary(both_wrong_model))
arm_rows <- grep("^study_arm", rownames(coef_mat), value = TRUE)
cat("\nContrasts vs Control (fixed effects; no other covariates):\n")
for (nm in arm_rows) {
  b <- coef_mat[nm, "Estimate"]
  se <- coef_mat[nm, "Std. Error"]
  t_col <- if ("t value" %in% colnames(coef_mat)) "t value" else "z value"
  p <- if ("Pr(>|t|)" %in% colnames(coef_mat)) {
    coef_mat[nm, "Pr(>|t|)"]
  } else {
    2 * pnorm(-abs(coef_mat[nm, t_col]))
  }
  cat(sprintf(
    "  %s: %.2f points [%.2f, %.2f], p = %.4f\n",
    nm, b, b - 1.96 * se, b + 1.96 * se, p
  ))
}

cat("\nEstimated marginal means:\n")
print(summary(emmeans(both_wrong_model, ~ study_arm), infer = c(TRUE, TRUE)))

Both-wrong rows: 430 ; participants: 309 

   Control    ChatGPT    Copilot Perplexity 
       147         63        140         80 

Rows by vignette:
   
    Control ChatGPT Copilot Perplexity
  A      41      21      58         21
  B      26      22      17         20
  C      13       0       3          6
  D      66      20      62         31
  E       1       0       0          2


boundary (singular) fit: see help('isSingular')



Linear mixed model fit by REML ['lmerMod']
Formula: delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette)
   Data: both_wrong

REML criterion at convergence: 3837.9

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.1915 -0.5440 -0.1178  0.5684  3.9862 

Random effects:
 Groups      Name        Variance Std.Dev.
 prolific_id (Intercept)  76.33    8.737  
 vignette    (Intercept)   0.00    0.000  
 Residual                386.65   19.663  
Number of obs: 430, groups:  prolific_id, 309; vignette, 5

Fixed effects:
                    Estimate Std. Error t value
(Intercept)            4.144      1.880   2.205
study_armChatGPT       9.261      3.371   2.748
study_armCopilot       9.229      2.675   3.450
study_armPerplexity    8.410      3.129   2.688

Correlation of Fixed Effects:
            (Intr) s_CGPT stdy_C
stdy_rmCGPT -0.558              
stdy_rmCplt -0.703  0.392       
stdy_rmPrpl -0.601  0.335  0.422
optimizer (nloptwrap) convergence code: 0 (OK)
boundar

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

Cannot use mode = "satterthwaite" because *lmerTest* package is not installed



 study_arm  emmean   SE  df asymp.LCL asymp.UCL z.ratio p.value
 Control      4.14 1.88 Inf      0.46      7.83   2.205  0.0275
 ChatGPT     13.41 2.80 Inf      7.92     18.89   4.791 <0.0001
 Copilot     13.37 1.90 Inf      9.64     17.10   7.027 <0.0001
 Perplexity  12.55 2.50 Inf      7.65     17.46   5.018 <0.0001

Degrees-of-freedom method: asymptotic 
Confidence level used: 0.95 


## Model A — unconditional confidence change

Every participant × vignette row. Unadjusted linear mixed model: `delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette)`, REML, Control as reference. No covariates. This comparison does not condition on whether the judgments were right or wrong.

In [15]:
print_arm_contrasts <- function(model, label) {
  cat("\n", label, "\n", sep = "")
  cat(
    "N participants:", n_distinct(model@frame$prolific_id),
    "; N observations:", nrow(model@frame), "\n\n"
  )
  print(summary(model))
  coef_mat <- coef(summary(model))
  arm_rows <- grep("^study_arm", rownames(coef_mat), value = TRUE)
  cat("\nContrasts vs Control:\n")
  for (nm in arm_rows) {
    b <- coef_mat[nm, "Estimate"]
    se <- coef_mat[nm, "Std. Error"]
    t_col <- if ("t value" %in% colnames(coef_mat)) "t value" else "z value"
    p <- if ("Pr(>|t|)" %in% colnames(coef_mat)) {
      coef_mat[nm, "Pr(>|t|)"]
    } else {
      2 * pnorm(-abs(coef_mat[nm, t_col]))
    }
    cat(sprintf(
      "  %s: %.2f points [%.2f, %.2f], p = %.4g\n",
      nm, b, b - 1.96 * se, b + 1.96 * se, p
    ))
  }
  cat("\nEstimated marginal means:\n")
  print(summary(emmeans(model, ~ study_arm), infer = c(TRUE, TRUE)))
  invisible(model)
}

model_a <- lmer(
  delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette),
  data = joint,
  REML = TRUE
)
print_arm_contrasts(model_a, "Model A — unconditional confidence change")


Model A — unconditional confidence change
N participants: 805 ; N observations: 4025 

Linear mixed model fit by REML ['lmerMod']
Formula: delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette)
   Data: joint

REML criterion at convergence: 35903.6

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-4.6155 -0.5070 -0.0537  0.5187  4.4342 

Random effects:
 Groups      Name        Variance Std.Dev.
 prolific_id (Intercept) 130.054  11.404  
 vignette    (Intercept)   5.519   2.349  
 Residual                355.900  18.865  
Number of obs: 4025, groups:  prolific_id, 805; vignette, 5

Fixed effects:
                    Estimate Std. Error t value
(Intercept)            6.010      1.441   4.171
study_armChatGPT       8.552      1.393   6.140
study_armCopilot       6.221      1.403   4.434
study_armPerplexity    8.645      1.429   6.049

Correlation of Fixed Effects:
            (Intr) s_CGPT stdy_C
stdy_rmCGPT -0.484              
stdy_rmCplt -0.481  0.498       
st

Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'pbkrtest.limit = 4025' (or larger)
[or, globally, 'set emm_options(pbkrtest.limit = 4025)' or larger];
but be warned that this may result in large computation time and memory use.

Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'lmerTest.limit = 4025' (or larger)
[or, globally, 'set emm_options(lmerTest.limit = 4025)' or larger];
but be warned that this may result in large computation time and memory use.



 study_arm  emmean   SE  df asymp.LCL asymp.UCL z.ratio p.value
 Control      6.01 1.44 Inf      3.19      8.83   4.171 <0.0001
 ChatGPT     14.56 1.44 Inf     11.74     17.38  10.118 <0.0001
 Copilot     12.23 1.45 Inf      9.39     15.07   8.440 <0.0001
 Perplexity  14.65 1.47 Inf     11.76     17.54   9.938 <0.0001

Degrees-of-freedom method: asymptotic 
Confidence level used: 0.95 


## Model B — both judgments wrong before consultation

A vignette is in this subgroup when the pre-consultation condition identification and the pre-consultation triage were both incorrect. That is a baseline split: `wrong_right` or `wrong_wrong` on both judgments. It does not require that the judgments stayed wrong after consultation. The same mixed model as Model A is fit inside this subgroup.

In [16]:
pre_wrong <- c("wrong_right", "wrong_wrong")
both_wrong_at_pre <- joint %>%
  filter(
    condition_transition %in% pre_wrong,
    triage_transition %in% pre_wrong
  )

cat(
  "Both wrong at pre — rows:", nrow(both_wrong_at_pre),
  "; participants:", n_distinct(both_wrong_at_pre$prolific_id), "\n"
)
cat("\nRows and participants by condition:\n")
both_wrong_at_pre %>%
  group_by(study_arm) %>%
  summarise(
    n = n(),
    participants = n_distinct(prolific_id),
    mean_delta = mean(delta_confidence),
    .groups = "drop"
  ) %>%
  print(n = Inf)

cat("\nRows by vignette:\n")
print(table(both_wrong_at_pre$vignette, both_wrong_at_pre$study_arm))

model_b <- lmer(
  delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette),
  data = both_wrong_at_pre,
  REML = TRUE
)
print_arm_contrasts(
  model_b,
  "Model B — both judgments wrong before consultation"
)

Both wrong at pre — rows: 1020 ; participants: 567 

Rows and participants by condition:
# A tibble: 4 × 4
  study_arm      n participants mean_delta
  <fct>      <int>        <int>      <dbl>
1 Control      255          140       6.45
2 ChatGPT      264          144      16.5 
3 Copilot      261          145      15.2 
4 Perplexity   240          138      17.0 

Rows by vignette:
   
    Control ChatGPT Copilot Perplexity
  A      79      85     100         74
  B      51      63      48         48
  C      23      18      15         15
  D     100      96      97         99
  E       2       2       1          4

Model B — both judgments wrong before consultation
N participants: 567 ; N observations: 1020 

Linear mixed model fit by REML ['lmerMod']
Formula: delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette)
   Data: both_wrong_at_pre

REML criterion at convergence: 9275.9

Scaled residuals: 
     Min       1Q   Median       3Q      Max 
-3.05512 -0.50942 -0.08887  0.5

Cannot use mode = "kenward-roger" because *pbkrtest* package is not installed

Cannot use mode = "satterthwaite" because *lmerTest* package is not installed



 study_arm  emmean   SE  df asymp.LCL asymp.UCL z.ratio p.value
 Control      5.95 2.85 Inf     0.363      11.5   2.087  0.0369
 ChatGPT     15.84 2.84 Inf    10.271      21.4   5.575 <0.0001
 Copilot     14.68 2.86 Inf     9.079      20.3   5.137 <0.0001
 Perplexity  16.98 2.87 Inf    11.357      22.6   5.922 <0.0001

Degrees-of-freedom method: asymptotic 
Confidence level used: 0.95 


## Figure — one confidence rating

Four panels, one rating. Points are observed means. Error bars are participant-clustered bootstrap intervals. Asterisks are mixed-model contrasts versus control from the models above.

- **A** All vignettes.
- **B** Both judgments wrong before consultation.
- **C** Both judgments stayed wrong. Exploratory, because this selects on the post-consultation result. ChatGPT contributes fewer vignettes here because more of its answers became correct.
- **D** Exactly one judgment stayed wrong. Exploratory for the same reason.

The panel in which neither judgment stayed wrong is not shown. Confidence rises there for an expected reason: some judgments became correct.


## Joint Table 1 — strata

One row per stratum and assigned condition. Means are observed. Differences versus control are from `delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette)`, REML. `neither_remained_incorrect` is the complement of the two post-consultation strata: neither judgment stayed wrong. It is not "both answers correct after consultation."

In [ ]:
contrast_frame <- function(model) {
  coef_mat <- coef(summary(model))
  arm_rows <- grep("^study_arm", rownames(coef_mat), value = TRUE)
  t_col <- if ("t value" %in% colnames(coef_mat)) "t value" else "z value"
  bind_rows(lapply(arm_rows, function(nm) {
    b <- coef_mat[nm, "Estimate"]
    se <- coef_mat[nm, "Std. Error"]
    p <- if ("Pr(>|t|)" %in% colnames(coef_mat)) {
      coef_mat[nm, "Pr(>|t|)"]
    } else {
      2 * pnorm(-abs(coef_mat[nm, t_col]))
    }
    data.frame(
      study_arm = sub("^study_arm", "", nm),
      diff_vs_control = b,
      std_error = se,
      ci_low = b - 1.96 * se,
      ci_high = b + 1.96 * se,
      p = p,
      stringsAsFactors = FALSE
    )
  }))
}

stratum_table <- function(data, stratum) {
  means <- data %>%
    group_by(study_arm) %>%
    summarise(
      n_vignettes = n(),
      n_participants = n_distinct(prolific_id),
      mean_delta = mean(delta_confidence),
      .groups = "drop"
    )
  model <- lmer(
    delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette),
    data = data,
    REML = TRUE
  )
  means %>%
    left_join(contrast_frame(model), by = "study_arm") %>%
    mutate(stratum = stratum, .before = 1)
}

pre_wrong_levels <- c("wrong_right", "wrong_wrong")
strata <- list(
  all_vignettes = joint,
  both_incorrect_before = joint %>%
    filter(
      condition_transition %in% pre_wrong_levels,
      triage_transition %in% pre_wrong_levels
    ),
  both_remained_incorrect = joint %>% filter(collapsed == "both_stayed_wrong"),
  exactly_one_remained_incorrect = joint %>% filter(collapsed == "one_stayed_wrong"),
  neither_remained_incorrect = joint %>% filter(collapsed == "neither_stayed_wrong")
)

joint_table_1 <- bind_rows(lapply(names(strata), function(nm) {
  stratum_table(strata[[nm]], nm)
}))

cat("\nJoint Table 1\n")
print(as.data.frame(joint_table_1), row.names = FALSE, digits = 4)

TABLE1_PATH <- file.path(ROOT, "output", "joint_confidence_table1_strata.csv")
write.csv(joint_table_1, TABLE1_PATH, row.names = FALSE)
cat("\nSaved", TABLE1_PATH, "\n")

## Joint Table 2 — robustness of the main analysis

Only the all-vignette model. The same three contrasts as Joint Table 1.

- Unadjusted: every vignette.
- Adjusted: age, sex (Female as reference), LLM-use frequency, and AI literacy. Items `aili_02`, `aili_05`, and `aili_11` are reverse-coded as `8 - value`. The sum is missing if any item is missing. Complete cases only.
- SA1: LLM participants with any transcript-validation category 4 excluded.
- SA2: those participants excluded, and LLM vignettes with category 2 or 3 also excluded. Control rows stay.

In [ ]:
AILI_ITEMS <- paste0("aili_", sprintf("%02d", 1:12))
REVERSE_AILI_ITEMS <- c("aili_02", "aili_05", "aili_11")
VALIDATION_COLS <- paste0("vig_", VIGNETTES, "_post_transcript_validation")
LLM_ARM_CODES <- c("1", "2", "3")

aili_recoded <- as.data.frame(lapply(AILI_ITEMS, function(col_name) {
  values <- as.numeric(df[[col_name]])
  if (col_name %in% REVERSE_AILI_ITEMS) values <- 8 - values
  values
}), stringsAsFactors = FALSE)
names(aili_recoded) <- AILI_ITEMS

person_covariates <- df %>%
  transmute(
    prolific_id = as.character(prolific_id),
    Age = as.numeric(Age),
    Sex = as.character(Sex),
    ai_frequency = as.numeric(ai_frequency),
    ai_literacy_sum = apply(aili_recoded, 1, function(row) {
      if (any(is.na(row))) NA_real_ else sum(row)
    })
  )

joint_cov <- joint %>%
  mutate(prolific_id_chr = as.character(prolific_id)) %>%
  left_join(person_covariates, by = c("prolific_id_chr" = "prolific_id")) %>%
  mutate(Sex = factor(Sex), Sex = relevel(Sex, ref = "Female"))

joint_adj <- joint_cov %>%
  filter(
    !is.na(Age),
    !is.na(Sex),
    !is.na(ai_frequency),
    !is.na(ai_literacy_sum)
  )

excluded_llm_ids <- function(validation_categories) {
  llm <- df %>% filter(as.character(study_arm) %in% LLM_ARM_CODES)
  vals <- llm %>% select(all_of(VALIDATION_COLS))
  flagged <- apply(vals, 1, function(row) {
    any(as.numeric(row) %in% validation_categories, na.rm = TRUE)
  })
  as.character(llm$prolific_id[flagged])
}

vignette_validation <- bind_rows(lapply(VIGNETTES, function(v) {
  df %>%
    transmute(
      prolific_id_chr = as.character(prolific_id),
      vignette = factor(v, levels = VIGNETTES),
      validation = as.numeric(.data[[paste0("vig_", v, "_post_transcript_validation")]])
    )
}))

sa1_drop_ids <- excluded_llm_ids(4)
joint_sa1 <- joint %>% filter(!as.character(prolific_id) %in% sa1_drop_ids)

joint_sa2 <- joint_sa1 %>%
  mutate(prolific_id_chr = as.character(prolific_id)) %>%
  left_join(vignette_validation, by = c("prolific_id_chr", "vignette")) %>%
  filter(is.na(validation) | !(validation %in% c(2, 3))) %>%
  select(-prolific_id_chr, -validation)

cat(
  "Adjusted sample:", nrow(joint_adj), "rows,",
  n_distinct(joint_adj$prolific_id), "participants\n"
)
cat(
  "SA1:", nrow(joint_sa1), "rows,",
  n_distinct(joint_sa1$prolific_id), "participants;",
  "excluded", length(sa1_drop_ids), "LLM participant(s) with category 4\n"
)
cat(
  "SA2:", nrow(joint_sa2), "rows,",
  n_distinct(joint_sa2$prolific_id), "participants\n"
)

fit_arm_model <- function(data, formula) {
  lmer(formula, data = data, REML = TRUE)
}

unadjusted_model <- fit_arm_model(
  joint,
  delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette)
)
adjusted_model <- fit_arm_model(
  joint_adj,
  delta_confidence ~ study_arm + Age + Sex + ai_frequency + ai_literacy_sum +
    (1 | prolific_id) + (1 | vignette)
)
sa1_model <- fit_arm_model(
  joint_sa1,
  delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette)
)
sa2_model <- fit_arm_model(
  joint_sa2,
  delta_confidence ~ study_arm + (1 | prolific_id) + (1 | vignette)
)

robust_models <- list(
  unadjusted = unadjusted_model,
  adjusted = adjusted_model,
  SA1 = sa1_model,
  SA2 = sa2_model
)
robust_n <- list(
  unadjusted = joint,
  adjusted = joint_adj,
  SA1 = joint_sa1,
  SA2 = joint_sa2
)

joint_table_2 <- bind_rows(lapply(names(robust_models), function(nm) {
  data <- robust_n[[nm]]
  contrast_frame(robust_models[[nm]]) %>%
    mutate(
      specification = nm,
      n_vignettes = nrow(data),
      n_participants = n_distinct(data$prolific_id),
      .before = 1
    )
}))

cat("\nJoint Table 2\n")
print(as.data.frame(joint_table_2), row.names = FALSE, digits = 4)

TABLE2_PATH <- file.path(ROOT, "output", "joint_confidence_table2_robustness.csv")
write.csv(joint_table_2, TABLE2_PATH, row.names = FALSE)
cat("\nSaved", TABLE2_PATH, "\n")